## Setup

Before extracting the training data and building the model, we need to ensure the config is set up correctly.
- `default_aoi` or default area of interest should be populated as a thin coastal strip (about 14 km max inland).
- `target` should be set to GMW ([projects/sat-io/open-datasets/GMW/annual-extent/GMW_MNG_2020](https://gee-community-catalog.org/projects/mangrove/#earth-engine-snippet-sentinel-raster-and-vector-baseline-v4019))
    - `mangrove_class`: 1
    - `permanent_water_threshold`: 100
- Within `export` settings,
    - `num_sample_points`: 6000
    - `target_name`: 'label'
    - `scale`: 10
    - `seed`: 48
    - `tile_scale`: 4
    - `geometries`: True
- `default_date_range` should have a `start_date`: '2020-01-01' and `end_date`: '2020-12-31'
- Within `validation` settings,
    - `pomeroon_river_aoi` should be defined with its 2D list of coordinates

In [13]:
import sys

import ee

from mangrove_mapper import MangroveMapper

sys.path.insert(0, "../../../")

PROJECT = "irp-zu"  # Insert your GEE project name

ee.Authenticate()
ee.Initialize(project=PROJECT)

mm = MangroveMapper(project_name=PROJECT)
CONFIG = {key: mm.config[key] for key in ['validation', 'target', 'export', 'default_date_range']}
CONFIG

{'validation': {'pomeroon_river_aoi': [[-58.793201027447836,
    7.641999643312983],
   [-58.786311616793554, 7.633424575768573],
   [-58.77487394309828, 7.621191216485879],
   [-58.76182767844984, 7.611492840798936],
   [-58.73607847190687, 7.600433021866128],
   [-58.70311796671348, 7.575930375898359],
   [-58.67016050315687, 7.5568715838076],
   [-58.6444112966139, 7.518070977770642],
   [-58.610422343977184, 7.473139628801602],
   [-58.57076856590101, 7.417820070012739],
   [-58.545191020735, 7.398584252947056],
   [-58.531458993179655, 7.384966344435998],
   [-58.51978657274669, 7.386327054087321],
   [-58.51120232824954, 7.414584961519535],
   [-58.53557798362562, 7.437395480797441],
   [-58.54931089378187, 7.460544404921525],
   [-58.60424253440687, 7.528622363801614],
   [-58.63994810081312, 7.576270580861263],
   [-58.69625303245375, 7.632080358980546],
   [-58.76217100120375, 7.652496753438371],
   [-58.79101011253187, 7.696558803331786],
   [-58.81273167189868, 7.67352708312

In [14]:
from pathlib import Path

import pandas as pd
import rasterio

from mangrove_mapper.inference import extent_tile, mosaic, predict_tile
from mangrove_mapper.models import MangroveClassifier
from mangrove_mapper.sensors import ALOSPalsar, Landsat8, Sentinel1, Sentinel2
from mangrove_mapper.utils.fusion import concat_sensors
from mangrove_mapper.utils.regions import tag_regions
from mangrove_mapper.validation.validator import auc_scores, evaluate_model, olofsson_estimate, summarise_estimates

REBUILD = False 

YEAR = 2021
DATA = Path("../../../data")
TRAINED = Path("../../../trained_models")
SHEET = Path("../../../mangrove_mapper/accuracy_assessment/"
             "region_2_accuracy_assessment_final.csv")

# region-2 (Pomeroon) 10 m feature stack + its water mask (clipped to aoi)
STACK = DATA / "feature_stack_region_2.tif"
GSW = DATA / "gsw_occurrence_10m_region_2.tif"

# export settings for the region-2 stack + GSW build
AOI = ee.Geometry.Polygon(CONFIG["validation"]["pomeroon_river_aoi"])
START, END = f"{YEAR}-01-01", f"{YEAR}-12-31"
CRS = "EPSG:32621"
SCALE = 10
TILE = 4096
DRIVE_FOLDER = "MangroveMapper_ML_Exports"

REFERENCES = ("ceo_label", "planet_label")

mc = MangroveClassifier()
print(f"REBUILD={REBUILD}")

REBUILD=False


## 1. Training data (WorldCover and GMW labels)

Here we need to export training data using both WorldCover and GMW labels for comparisons. Onece those training sets are obtained, subsets of the different sensors can be obtained for comparisons as well. 

There should be 35 feature bands and just under 12,000 rows (~6,000 per sample class) for both training sets

In [15]:
GMW_CSV = DATA / "training_data_coastwide.csv"
WC_CSV = DATA / "training_data_coastwide_wc.csv"

if REBUILD or not WC_CSV.exists():
    # the config target, so the WorldCover export temporarily swaps the config target
    # (class 95) before restoring the GMW default (class 1).
    tgt = mm.config['target']
    saved = dict(tgt)
    tgt['dataset_id'], tgt['mangrove_class'] = 'ESA/WorldCover/v100', 95
    mm.get_sensor_data(WC_CSV.stem, retrieve_from_drive=True, presence_only=False)
    tgt.update(saved)   # restore GMW default for the export below

if REBUILD or not GMW_CSV.exists():
    # GMW is presence-only and matches the config target defaults
    mm.get_sensor_data(GMW_CSV.stem, retrieve_from_drive=True, presence_only=True)

regions_geojson = (DATA / "guyana_regions.geojson") if (DATA / "guyana_regions.geojson").exists() else None
train_wc = tag_regions(pd.read_csv(WC_CSV), regions_data_path=regions_geojson)
train_gmw = tag_regions(pd.read_csv(GMW_CSV), regions_data_path=regions_geojson)

FEATS = [c for c in train_wc.columns if c not in mc.NON_FEATURES]
LBAND = tuple(c for c in FEATS if c.startswith("alos_"))  # ALOS PALSAR L-band SAR
CBAND = tuple(c for c in FEATS if c.startswith("s1_"))    # Sentinel-1 C-band SAR
DROPS = {"": (), "L": LBAND, "C": CBAND, "SAR": LBAND + CBAND}
print(f"WC {len(train_wc)} rows | GMW {len(train_gmw)} rows | {len(FEATS)} features")
print(f"L-band {len(LBAND)} bands | C-band {len(CBAND)} bands")

/opt/miniconda3/envs/irp2026/lib/python3.14/site-packages/geopandas/array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
/opt/miniconda3/envs/irp2026/lib/python3.14/site-packages/geopandas/array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


WC 11992 rows | GMW 11992 rows | 35 features
L-band 4 bands | C-band 5 bands


## 2. Region-2 feature stack and water mask

We now need a contiguous feature stack and water mask for model inference in 2021. This stack and water mask need to be clipped to the Region 2 AOI.

In [16]:
def region_stack():
    sensors = [Landsat8(START, END, AOI, PROJECT), Sentinel2(START, END, AOI, PROJECT),
               Sentinel1(START, END, AOI, PROJECT), ALOSPalsar(START, END, AOI, PROJECT)]
    return concat_sensors(sensors).toFloat().select(FEATS).clip(AOI)


def gsw_image():
    return (ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence")
            .unmask(0).clip(AOI).toUint8())


if REBUILD or not STACK.exists():
    tiles = mm.export_image_tiled(region_stack(), STACK.stem, scale=SCALE, crs=CRS,
                                  aoi=AOI, tile=TILE, folder=DRIVE_FOLDER, output_dir=DATA)
    mosaic(sorted(tiles), STACK, nodata=None)

if REBUILD or not GSW.exists():
    tiles = mm.export_image_tiled(gsw_image(), GSW.stem, scale=SCALE, crs=CRS,
                                  aoi=AOI, tile=TILE, folder=DRIVE_FOLDER, output_dir=DATA)
    mosaic(sorted(tiles), GSW, nodata=0)

with rasterio.open(STACK) as s:
    print(f"stack {s.width} x {s.height} px, {s.count} bands, {s.crs}")

stack 3317 x 3458 px, 35 bands, EPSG:32621


## 3. Models and probability rasters

With all stack and mask in place as well as the training datasets, we have all we need to train models and test out various experiments. This cell serves as a registry that drives all the various studies such as the "ablation" study with various subsets of sensors, the head-to-head between estimators (random forest and xgboost) and the comparison between training labels (worldcover and global mangrove watch).

NOTE: Please keep in mind that if this is run for the first time, six models will need to be trained using RandomizedSearchCV so it may take some time (at least 50 mins in my experience)

If you'd like to avoid the training wait times, you can download the trained models below and move them to `ROOT_DIR/trained_models/`  

[Download trained_models!](https://imperiallondon-my.sharepoint.com/:f:/g/personal/zam25_ic_ac_uk/IgBgU0TXjuEFR63aTh0tIGq1AZOXZ-CILCBPFWHe0s1IRgY?e=4pFA1U)


In [17]:
MODELS = [
    # ablation - Random Forest, WorldCover labels, Threshold 70
    dict(section="Ablation (Random Forest)", label="Full fusion", estimator="random_forest", data="wc", drop="",    prob="2021-prob-rf.tif",           save="random_forest_6k_full_fusion",    t=0.70),
    dict(section="Ablation (Random Forest)", label="Without L-band",    estimator="random_forest", data="wc", drop="L",   prob="2021-prob-rf-without_l_band.tif", save="random_forest_6k_without_l_band", t=0.70),
    dict(section="Ablation (Random Forest)", label="Without C-band",    estimator="random_forest", data="wc", drop="C",   prob="2021-prob-rf-without_c_band.tif", save="random_forest_6k_without_c_band", t=0.70),
    dict(section="Ablation (Random Forest)", label="Without SAR",    estimator="random_forest", data="wc", drop="SAR", prob="2021-prob-rf-without_sar.tif",    save="random_forest_6k_without_sar",    t=0.70),
    
    # estimator - full fusion, WorldCover labels, T70. The RF baseline is the "Full fusion" row above (same map), so only XGBoost is shown here.
    dict(section="Estimator", label="XGBoost", estimator="xgboost", data="wc", drop="", prob="2021-prob-xgb.tif", save="xgb_6k_full_fusion", t=0.70),
    # training labels - GMW vs WorldCover, XGBoost. 
    # the WorldCover-labelled XGB is added AUC-only in the table cell.
    dict(section="Training labels", label="XGB-GMW", estimator="xgboost", data="gmw", drop="", prob="2021-prob-xgb-gmw.tif", save="xgb_6k_gmw_full_fusion", t=0.50)
]

def ensure_prob(config):
    prob = DATA / config["prob"]
    if not REBUILD and prob.exists():
        return prob
    data = train_wc if config["data"] == "wc" else train_gmw
    drop = DROPS[config["drop"]]
    feats = [c for c in data.columns if c not in mc.NON_FEATURES and c not in drop]
    X, y, groups = mc.features_except(data, drop), data["label"].values, data["region"].values
    dist = mc.RF_PARAMS_DIST if config["estimator"] == "random_forest" else mc.XGB_PARAMS_DIST
    model, params = mc.build_final_model(X, y, groups=groups,
                                         base_estimator=config["estimator"], param_dist=dist)
    mc.save_model(model, name=config["save"], best_params=params)
    predict_tile(model, feats, STACK, prob)
    return prob

import time

builds = list(dict.fromkeys(DATA / c["prob"] for c in MODELS))
for i, prob_path in enumerate(builds, 1):
    config = next(c for c in MODELS if DATA / c["prob"] == prob_path)
    if not REBUILD and prob_path.exists():
        print(f"[{i}/{len(builds)}] reuse  {prob_path.name}", flush=True)
        continue
    print(f"[{i}/{len(builds)}] train  {config['estimator']:13s} {config['label']:12s} => {prob_path.name} ...", flush=True)
    t0 = time.time()
    ensure_prob(config)
    print(f"          done in {time.time() - t0:,.0f}s", flush=True)

[1/6] reuse  2021-prob-rf.tif
[2/6] reuse  2021-prob-rf-without_l_band.tif
[3/6] reuse  2021-prob-rf-without_c_band.tif
[4/6] reuse  2021-prob-rf-without_sar.tif
[5/6] reuse  2021-prob-xgb.tif
[6/6] reuse  2021-prob-xgb-gmw.tif


## 4. Extent maps

Applying a probability threshold to each probability raster to build an extent map with spectral and water masks.

In [18]:
def ensure_extent(config):
    prob = DATA / config["prob"]
    tag = int(config["t"] * 100)
    ext = DATA / f"{prob.stem.replace('2021-prob', '2021-extent')}-T{tag}.tif"
    if REBUILD or not ext.exists():
        ha = extent_tile(prob, STACK, GSW, config["t"], ext)
        print(f"  built {ext.name}: {ha:,.0f} ha mangrove")
    return ext

for config in MODELS:
    config["extent"] = ensure_extent(config)

## 5. Design-based accuracy and AUC

We now have an extent map by which we can use the olofsson estimator to determine user's, producer's, overall accuracy, error-adjusted area estimates and the various threshold-independent areas under the curve (AUC) metrics.

In [19]:
results, aucs = {}, {}
for config in MODELS:
    col = config["extent"].stem
    df_eval, meta = evaluate_model(str(SHEET), model_name=col,
                                   model_extent_map_path=str(config["extent"]))
    for ref in REFERENCES:
        results[(config["section"], config["label"], ref)] = olofsson_estimate(
            df_eval, map_col=col, weights=meta["weights"],
            domain_ha=meta["domain_ha"], ref_col=ref)
    if config["prob"] not in aucs:
        aucs[config["prob"]] = auc_scores(str(SHEET), str(DATA / config["prob"]), ref_cols=REFERENCES)

# quick per-section preview (CEO reference)
for section in ("Ablation (RF)", "Estimator", "Training labels"):
    print(section)
    display(summarise_estimates(
        {config["label"]: results[(section, config["label"], "ceo_label")]
         for config in MODELS if config["section"] == section}, focus="mangrove"))

Ablation (RF)


""


Estimator


,n,overall acc.,mangrove UA,mangrove PA,mangrove area (ha)
XGBoost,400,0.902 ± 0.023,0.723 ± 0.061,0.955 ± 0.048,4642.869 ± 441.802


Training labels


,n,overall acc.,mangrove UA,mangrove PA,mangrove area (ha)
XGB-GMW,400,0.965 ± 0.017,0.919 ± 0.042,0.933 ± 0.055,4592.101 ± 333.679


## 6. Report Figure

latex table that appears in the final report

In [20]:
# one table, three sections, both references.
def pm(v, ci, fmt="{:.3f}"):
    return f"{fmt.format(v)} ± {fmt.format(ci)}"

rows = []
for config in MODELS:
    
    if config["section"] == "Training labels":
        for ref in REFERENCES:
            a = aucs["2021-prob-xgb.tif"][ref]
            rows.append({
                "section": "Training labels", "Map": "XGB-WorldCover",
                "Reference": ref.replace("_label", ""), "n": 400,
                "OA": "", "UA": "", "PA": "", "Area (ha)": "",
                "ROC-AUC": f"{a['roc_auc']:.3f}", "PR-AUC": f"{a['pr_auc']:.3f}"})
    for ref in REFERENCES:
        r = results[(config["section"], config["label"], ref)]
        pc = r["per_class"].loc["mangrove"]
        a = aucs[config["prob"]][ref]
        rows.append({
            "section": config["section"], "Map": config["label"],
            "Reference": ref.replace("_label", ""), "n": r["n"],
            "OA": pm(r["overall_accuracy"], r["overall_accuracy_ci"]),
            "UA": pm(pc["users_acc"], pc["users_ci"]),
            "PA": pm(pc["producers_acc"], pc["producers_ci"]),
            "Area (ha)": pm(pc["area_ha"], pc["area_ha_ci"], "{:,.0f}"),
            "ROC-AUC": f"{a['roc_auc']:.3f}", "PR-AUC": f"{a['pr_auc']:.3f}"})

# Chan-Bagot (2024) is an independent 30 m data-fusion map of the same estuary.
rows.append({"section": "Training labels", "Map": "Chan-Bagot et al. (2024)", "Reference": "",
             "n": 401, "OA": "0.950", "UA": "0.690", "PA": "0.870",
             "Area (ha)": "3,189 ± 707", "ROC-AUC": "", "PR-AUC": ""})

table = pd.DataFrame(rows)
table

,section,Map,Reference,n,OA,UA,PA,Area (ha),ROC-AUC,PR-AUC
0,Ablation (Random Forest),Full fusion,ceo,400,0.918 ± 0.023,0.774 ± 0.060,0.928 ± 0.058,"4,630 ± 439",0.964,0.942
1,Ablation (Random Forest),Full fusion,planet,400,0.892 ± 0.028,0.763 ± 0.061,0.842 ± 0.074,"5,036 ± 553",0.960,0.940
2,Ablation (Random Forest),Without L-band,ceo,400,0.895 ± 0.026,0.731 ± 0.062,0.882 ± 0.071,"4,643 ± 510",0.958,0.940
3,Ablation (Random Forest),Without L-band,planet,400,0.882 ± 0.029,0.736 ± 0.062,0.834 ± 0.077,"4,945 ± 569",0.955,0.935
4,Ablation (Random Forest),Without C-band,ceo,400,0.919 ± 0.024,0.800 ± 0.059,0.892 ± 0.066,"4,785 ± 470",0.955,0.924
5,Ablation (Random Forest),Without C-band,planet,400,0.893 ± 0.029,0.789 ± 0.060,0.814 ± 0.075,"5,177 ± 570",0.951,0.915
6,Ablation (Random Forest),Without SAR,ceo,400,0.892 ± 0.028,0.757 ± 0.062,0.841 ± 0.075,"4,921 ± 549",0.945,0.912
7,Ablation (Random Forest),Without SAR,planet,400,0.870 ± 0.032,0.751 ± 0.062,0.778 ± 0.078,"5,283 ± 623",0.942,0.905
8,Estimator,XGBoost,ceo,400,0.902 ± 0.023,0.723 ± 0.061,0.955 ± 0.048,"4,643 ± 442",0.964,0.949
9,Estimator,XGBoost,planet,400,0.885 ± 0.027,0.723 ± 0.061,0.890 ± 0.067,"4,987 ± 531",0.961,0.944


In [21]:
# assemble regional table (point estimate ± 95% CI)
COLS = ["Map", "Reference", "OA", "UA", "PA", "Area (ha)", "ROC-AUC", "PR-AUC"]

def cell(x):
    return ("" if x is None else str(x)).replace("±", r"$\pm$").replace("—", r"---")

# highlight the best model (XGB-GMW)
BOLD_MAPS = {"XGB-GMW"}

body = []
for section in ("Ablation (Random Forest)", "Estimator", "Training labels"):
    body.append(r"\midrule")
    body.append(rf"\multicolumn{{{len(COLS)}}}{{l}}{{\textit{{{section}}}}} \\")
    for _, row in table[table.section == section].iterrows():
        cells = [cell(row[c]) for c in COLS]
        if row["Map"] in BOLD_MAPS:
            cells = [rf"\textbf{{{c}}}" if c else c for c in cells]
        body.append(" & ".join(cells) + r" \\")

latex = "\n".join([
    r"\begin{table}[htbp]", r"\centering",
    r"\caption{Pomeroon River Estuary (Region 2) extent-map model comparisons from 400 ",
    r"stratified independent reference plots interpreted against CEO and Planet imagery. \textcite{Chan2024} 30 m extent map serves as a suggestive benchmark.} ",
    r"\label{tab:region2-results}",
    r"\begin{adjustbox}{max width=\textwidth}",
    r"\begin{tabular}{llllllrr}",
    r"\toprule",
    " & ".join(COLS) + r" \\",
    *body,
    r"\bottomrule",
    r"\end{tabular}", r"\end{adjustbox}", r"\end{table}"])

Path("region2_results.tex").write_text(latex)
print(latex)

\begin{table}[htbp]
\centering
\caption{Pomeroon River Estuary (Region 2) extent-map model comparisons from 400 
stratified independent reference plots interpreted against CEO and Planet imagery. \textcite{Chan2024} 30 m extent map serves as a suggestive benchmark.} 
\label{tab:region2-results}
\begin{adjustbox}{max width=\textwidth}
\begin{tabular}{llllllrr}
\toprule
Map & Reference & OA & UA & PA & Area (ha) & ROC-AUC & PR-AUC \\
\midrule
\multicolumn{8}{l}{\textit{Ablation (Random Forest)}} \\
Full fusion & ceo & 0.918 $\pm$ 0.023 & 0.774 $\pm$ 0.060 & 0.928 $\pm$ 0.058 & 4,630 $\pm$ 439 & 0.964 & 0.942 \\
Full fusion & planet & 0.892 $\pm$ 0.028 & 0.763 $\pm$ 0.061 & 0.842 $\pm$ 0.074 & 5,036 $\pm$ 553 & 0.960 & 0.940 \\
Without L-band & ceo & 0.895 $\pm$ 0.026 & 0.731 $\pm$ 0.062 & 0.882 $\pm$ 0.071 & 4,643 $\pm$ 510 & 0.958 & 0.940 \\
Without L-band & planet & 0.882 $\pm$ 0.029 & 0.736 $\pm$ 0.062 & 0.834 $\pm$ 0.077 & 4,945 $\pm$ 569 & 0.955 & 0.935 \\
Without C-band & ceo & 0.9